In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Generator Model
class Generator(nn.Module):
    def __init__(self, input_dim=784, output_dim=784):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, output_dim),
            nn.Tanh()  
        )

    def forward(self, z):
        img = self.model(z)
        return img  

# Discriminator Model
class Critic(nn.Module):
    def __init__(self, input_dim=784):
        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)  
        )

    def forward(self, img):
        validity = self.model(img)
        return validity


In [ ]:
# Computes the gradient penalty for WGAN-GP
def compute_gradient_penalty(D, real_samples, fake_samples):
    alpha = torch.rand(real_samples.size(0), 1).to(real_samples.device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)

    d_interpolates = D(interpolates)
    fake = torch.ones(d_interpolates.shape, requires_grad=False).to(real_samples.device)

    gradients = autograd.grad(outputs=d_interpolates, inputs=interpolates,
                              grad_outputs=fake, create_graph=True,
                              retain_graph=True, only_inputs=True)[0]
    
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


In [ ]:
# Hyperparameters
batch_size = 64
latent_dim = 64  
n_epochs = 600
lambda_gp = 10  # Gradient penalty weight
lr = 0.0001
n_critic = 5  # Number of critic updates per generator update

# Prepare DataLoader for MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
dataloader = DataLoader(torchvision.datasets.MNIST(root="./data", train=True, transform=transform, download=True),
                        batch_size=batch_size, shuffle=True)

# Initialize models
G = Generator(input_dim=latent_dim, output_dim=784).cuda()
D = Critic(input_dim=784).cuda()

# Optimizers
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.9))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.9))

In [ ]:
# Training loop
for epoch in range(n_epochs):
    for i, (real_imgs, _) in enumerate(dataloader):

        real_imgs = real_imgs.view(real_imgs.size(0), -1).cuda()  # Flatten MNIST images to 784
        batch_size = real_imgs.size(0)

        for _ in range(n_critic):
            z = torch.randn(batch_size, latent_dim).cuda()  # Sample Gaussian noise
            fake_imgs = G(z).detach()  # Generate fake images

            real_validity = D(real_imgs)
            fake_validity = D(fake_imgs)
            gradient_penalty = compute_gradient_penalty(D, real_imgs, fake_imgs)
            
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty

            optimizer_D.zero_grad()
            d_loss.backward()
            optimizer_D.step()

        if i % n_critic == 0:
            z = torch.randn(batch_size, latent_dim).cuda()
            gen_imgs = G(z)
            g_loss = -torch.mean(D(gen_imgs))

            optimizer_G.zero_grad()
            g_loss.backward()
            optimizer_G.step()

    print(f"Epoch [{epoch}/{n_epochs}] | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

    # Save generated images every few epochs
    if epoch % 10 == 0:
        batch_size_real = gen_imgs.shape[0]  # Get actual batch size
        torchvision.utils.save_image(gen_imgs.view(batch_size_real, 1, 28, 28),
                                    f"generated_epoch_{epoch}.png", normalize=True)


In [ ]:
# Save trained models
torch.save(G.state_dict(), './G_WGANgp.pth')
torch.save(D.state_dict(), './D_WGANgp.pth')

In [ ]:
# check the generator for generating MNIST image
test_image = torch.randn(1, 64).cuda()
test_G_image = G(test_image)
print(test_G_image.shape)

In [ ]:
# View 100 images by the generator
z = torch.randn(100, latent_dim).cuda()
fake_images = G(z)
print(fake_images.shape)
fake_images = fake_images.reshape(fake_images.size(0), 1, 28, 28)
fake_images = fake_images.cpu().data

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(fake_images[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# G.load_state_dict(torch.load('./G_WGANgp.pth'))
# print(G)
